# Deepfake Detection Training Pipeline
**Target Hardware**: 2x NVIDIA T4 GPUs (`DataParallel`, `fp16` mixed precision)

### Tier 2 Specifications (Research Benchmark Scale)
- **Class Balance**: 1,000 Real vs 4,000 Fake videos (125,000 frame crops total across 5 manipulation types)
- **Partitioning**: GroupKFold by Video-ID (0% identity leakage via NetworkX connected components)
- **Extraction**: 1.30x bounding box scaling factor saved as Lossless WebP (`.webp`)
- **Frequency Stream**: 2-Channel 2D Real FFT (Log-Magnitude + Normalized Phase Angle $\theta / \pi$)
- **Augmentations**: Geometric (affine, flip), color jitter, downscale, and Gaussian blur
- **Model**: ConvNeXt-Base (1024-d) + 2D FFT Log/Phase Spectrum (128-d) cross-attention fusion
- **Training**: 2-phase fine-tuning (3 epochs head warmup, 15 epochs 10-group LLRD differential LR)
- **Sequence Length**: $T = 16$ consecutive frames at 30 FPS for temporal continuity
- **Export**: PyTorch `.pth` checkpoint and ONNX opset 14 (`.onnx`)

In [ ]:
# Install dependencies and verify environment
!pip install "Pillow<11.0" timm facenet-pytorch albumentations grad-cam onnx onnxruntime pyyaml -q

import os, sys, re, random, time, cv2, torch, shutil, copy, numpy as np

# Clone repository if running in cloud environment (Kaggle/Colab)
if not os.path.exists("src") and not os.path.exists("deepfake-detection"):
    !git clone https://github.com/yyouretoast/deepfake-detection.git
    if os.path.exists("deepfake-detection"):
        sys.path.append(os.path.abspath("deepfake-detection"))
elif os.path.exists("deepfake-detection"):
    sys.path.append(os.path.abspath("deepfake-detection"))

import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from tqdm import tqdm
import timm
from facenet_pytorch import MTCNN
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from accelerate import Accelerator, notebook_launcher

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 1. Setup & Configuration

In [ ]:
# Load configuration and set random seeds
try:
    from src.config import load_config
    CFG = load_config()
except Exception:
    CFG = {
        'paths': {'kaggle_input': '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23', 'output_dir': '/kaggle/working/frames_v2'},
        'preprocessing': {'img_size': 512, 'padding_scale': 1.30, 'max_real_videos': 1000, 'max_fake_per_dir': 1000, 'frames_per_video': 25},
        'training': {'batch_size': 32, 'epochs_phase1': 3, 'epochs_phase2': 15, 'lr_phase1': 1e-4, 'lr_backbone': 1e-5, 'lr_head': 1e-4, 'seed': 42},
        'manipulation_types': {'all': ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter'], 'held_out_loto': 'FaceShifter'}
    }

SEED = CFG.get('training', {}).get('seed', 42)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

BASE = CFG.get('paths', {}).get('kaggle_input', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23')
OUTPUT_DIR = "/kaggle/working/frames_v2"
IMG_SIZE = CFG.get('preprocessing', {}).get('img_size', 512)
PADDING_SCALE = CFG.get('preprocessing', {}).get('padding_scale', 1.30)
FAKE_DIRS = CFG.get('manipulation_types', {}).get('all', ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter'])
HELD_OUT_TYPE = CFG.get('manipulation_types', {}).get('held_out_loto', 'FaceShifter')

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(f"{OUTPUT_DIR}/real", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/fake", exist_ok=True)
print("Configuration loaded.")

## 2. Face Extraction

In [ ]:
# Extract face crops using GPU MTCNN
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
mtcnn = MTCNN(keep_all=True, post_process=False, device=device, select_largest=True)

def extract_video(v_path, out_dir, folder_name, v_name, frames_per_video=15):
    if not os.path.exists(v_path): return 0
    cap = cv2.VideoCapture(v_path)
    if not cap.isOpened(): return 0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return 0
    actual = min(frames_per_video, total)
    step = max(total // actual, 1)
    target_frames = set(i * step for i in range(actual))
    frames_pil, frames_rgb = [], []
    curr_frame = 0
    max_target = max(target_frames)
    while cap.isOpened() and len(frames_rgb) < actual and curr_frame <= max_target:
        if curr_frame in target_frames:
            ret, frame = cap.read()
            if ret and frame is not None:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames_rgb.append(rgb)
                frames_pil.append(Image.fromarray(rgb))
        else:
            cap.grab()
        curr_frame += 1
    cap.release()
    if not frames_pil: return 0
    try:
        boxes_list, _ = mtcnn.detect(frames_pil)
        if boxes_list is None: boxes_list = [None] * len(frames_pil)
    except Exception:
        boxes_list = [None] * len(frames_pil)
    saved = 0
    for idx, (rgb, boxes) in enumerate(zip(frames_rgb, boxes_list)):
        if boxes is None or len(boxes) == 0: continue
        h, w, _ = rgb.shape
        best_box = max(boxes, key=lambda b: (b[2]-b[0])*(b[3]-b[1]))
        x1, y1, x2, y2 = best_box[:4]
        bw, bh = x2 - x1, y2 - y1
        if bw < 10 or bh < 10: continue
        cx, cy = (x1 + x2)/2.0, (y1 + y2)/2.0
        nbw, nbh = bw * PADDING_SCALE, bh * PADDING_SCALE
        nx1, ny1 = max(0, int(cx - nbw/2.0)), max(0, int(cy - nbh/2.0))
        nx2, ny2 = min(w, int(cx + nbw/2.0)), min(h, int(cy + nbh/2.0))
        face = rgb[ny1:ny2, nx1:nx2]
        if face.size == 0 or face.shape[0] < 10 or face.shape[1] < 10: continue
        face_resized = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
        fname = f"{folder_name}_{v_name}_f{idx}.webp"
        cv2.imwrite(os.path.join(out_dir, fname), cv2.cvtColor(face_resized, cv2.COLOR_RGB2BGR))
        saved += 1
    return saved

if os.path.exists(BASE):
    df_path = os.path.join(BASE, "Deepfakes")
    fake_videos = sorted([f for f in os.listdir(df_path) if f.endswith('.mp4')])[:150]
    
    real_ids = set()
    for f in fake_videos:
        match = re.search(r'(\d+)_(\d+)', f)
        if match:
            real_ids.add(match.group(1))
            real_ids.add(match.group(2))
    
    real_path = os.path.join(BASE, "original")
    real_count = 0
    for rid in tqdm(sorted(list(real_ids)), desc="Extracting Real Identities"):
        v_path = os.path.join(real_path, f"{rid}.mp4")
        real_count += extract_video(v_path, f"{OUTPUT_DIR}/real", "original", rid)
    print(f"Extracted {real_count} real frames.")
    
    for fd in FAKE_DIRS:
        fd_path = os.path.join(BASE, fd)
        if not os.path.exists(fd_path): continue
        fake_count = 0
        for f in tqdm(fake_videos, desc=f"Extracting {fd}"):
            v_path = os.path.join(fd_path, f)
            v_name = os.path.splitext(f)[0]
            fake_count += extract_video(v_path, f"{OUTPUT_DIR}/fake", fd, v_name)
        print(f"Extracted {fake_count} frames for {fd}.")

## 3. Video-ID GroupKFold Partitioning

In [ ]:
# Identity-safe split — imported from src/ (single source of truth)
# Uses graph-connected component partitioning to guarantee zero identity leakage.
# Note: this replaces the previously-inline implementation with the same algorithm.
from src.dataset.loader import perform_graph_split, extract_identities, extract_video_id

# Copy extracted frames to Linux RAM disk (/tmp/ram_frames) for zero-latency I/O
RAM_DIR = "/tmp/ram_frames"
if os.path.exists(OUTPUT_DIR) and not os.path.exists(RAM_DIR):
    print("Copying dataset to Linux RAM Disk (/tmp/ram_frames)...")
    shutil.copytree(OUTPUT_DIR, RAM_DIR, dirs_exist_ok=True)
    print("RAM Disk Copy Complete.")

DATASET_DIR = RAM_DIR if os.path.exists(RAM_DIR) else OUTPUT_DIR
real_files = [(os.path.join(DATASET_DIR, "real", f), 0) for f in os.listdir(f"{DATASET_DIR}/real")] if os.path.exists(f"{DATASET_DIR}/real") else []
fake_files = [(os.path.join(DATASET_DIR, "fake", f), 1) for f in os.listdir(f"{DATASET_DIR}/fake")] if os.path.exists(f"{DATASET_DIR}/fake") else []
all_samples = real_files + fake_files

print(f"Total Real Files: {len(real_files)} | Total Fake Files: {len(fake_files)}")
train_samples, val_samples, test_samples = perform_graph_split(all_samples, seed=SEED)
# all_samples saved before split — needed by LOTO benchmark (Cell 17)

print(f"Train samples: {len(train_samples)} | Val samples: {len(val_samples)} | Test samples: {len(test_samples)}")

## 4. PyTorch Dataset & Augmentations

In [ ]:
# PyTorch Dataset and DataLoader initialization
# Single mild downscale augmentation — teaches resolution robustness without
# double-compressing FF++ C23 (already JPEG-compressed), which would teach
# the model to detect compression artifacts instead of manipulation artifacts
# and hurt cross-dataset generalization (Celeb-DF v2 / DFDC).
try:
    downscale_comp = A.Downscale(scale_range=(0.7, 0.9), p=0.1)
except Exception:
    downscale_comp = A.Downscale(scale_min=0.7, scale_max=0.9, p=0.1)
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.5),
    A.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10, p=0.5),
    downscale_comp,
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# SequenceVideoDataset — imported from src/ (single source of truth)
from src.dataset.loader import group_samples_by_video, perform_graph_split, SequenceVideoDataset

video_samples = group_samples_by_video(all_samples)
train_vids, val_vids, test_vids = perform_graph_split(video_samples, seed=SEED)

seq_len = CFG.get('training', {}).get('seq_len', 8)
train_ds = SequenceVideoDataset(train_vids, transform=train_transform, seq_len=seq_len)
val_ds = SequenceVideoDataset(val_vids, transform=eval_transform, seq_len=seq_len)
test_ds = SequenceVideoDataset(test_vids, transform=eval_transform, seq_len=seq_len)

batch_sz = CFG.get('training', {}).get('batch_size', 32)
train_labels = [v[1] for v in train_vids]
class_counts = np.maximum(np.bincount(train_labels), 1)
class_weights = 1. / class_counts
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=batch_sz, sampler=sampler, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_sz, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_sz, shuffle=False, num_workers=4, pin_memory=True)

print(f"DataLoaders ready | Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)} | Test Batches: {len(test_loader)}")

## 5. Dual-Stream Model Architecture

In [ ]:
# ── Model Architecture ───────────────────────────────────────────────────────
# Single source of truth: src/models/hybrid_detector.py
# This replaces the previously-inline model definition which had diverged from
# src/ (missing attn_out_proj, hardcoded autocast device, no LoRA/temporal support).
#
# IMPORTANT: This cell uses the src/ architecture. The checkpoint produced by
# this notebook will be incompatible with checkpoints trained on the old inline
# architecture. If you have an existing checkpoint, retrain from scratch.
# ─────────────────────────────────────────────────────────────────────────────
from src.models.hybrid_detector import HybridDeepfakeDetector, build_model

model = build_model(use_fft=True, device=device, pretrained=True, config=CFG)
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model initialized on {device}")
print(f"Total parameters:     {n_params:,} ({n_params/1e6:.1f}M)")
print(f"Trainable parameters: {n_trainable:,} ({n_trainable/1e6:.1f}M)")


### 5.1 2D Real FFT Frequency Spectrum Visualizer
Visualizes centered log-magnitude frequency spectra (`torch.fft.fftshift`) comparing real face crops vs deepfake face crops to illustrate high-frequency grid artifacts.

In [ ]:
# Visualizing 2D Real FFT Centered Log-Magnitude Spectrums
try:
    if len(val_ds) > 0:
        real_sample = None
        fake_sample = None
        for imgs_seq, label in val_ds:
            img = imgs_seq[0] if imgs_seq.ndim == 4 else imgs_seq
            if label == 0 and real_sample is None:
                real_sample = img
            elif label == 1 and fake_sample is None:
                fake_sample = img
            if real_sample is not None and fake_sample is not None:
                break
        
        if real_sample is not None and fake_sample is not None:
            def compute_fft_spectrum(img_tensor):
                # img_tensor: [3, H, W] normalized
                gray = 0.299 * img_tensor[0] + 0.587 * img_tensor[1] + 0.114 * img_tensor[2]
                fft2d = torch.fft.rfft2(gray, norm="ortho")
                mag = torch.abs(fft2d)
                centered = torch.fft.fftshift(mag, dim=-2)
                log_mag = torch.log(centered + 1e-5)
                return log_mag.cpu().numpy()

            real_fft = compute_fft_spectrum(real_sample)
            fake_fft = compute_fft_spectrum(fake_sample)

            fig, axes = plt.subplots(2, 2, figsize=(10, 8))
            # Unnormalize images for display
            mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
            std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
            r_img = (real_sample.cpu().numpy() * std + mean).clip(0, 1).transpose(1, 2, 0)
            f_img = (fake_sample.cpu().numpy() * std + mean).clip(0, 1).transpose(1, 2, 0)

            axes[0, 0].imshow(r_img)
            axes[0, 0].set_title("Real Face (RGB)")
            axes[0, 0].axis("off")

            im0 = axes[0, 1].imshow(real_fft, cmap="viridis")
            axes[0, 1].set_title("Real Face 2D FFT Spectrum")
            axes[0, 1].axis("off")
            fig.colorbar(im0, ax=axes[0, 1])

            axes[1, 0].imshow(f_img)
            axes[1, 0].set_title("Deepfake Face (RGB)")
            axes[1, 0].axis("off")

            im1 = axes[1, 1].imshow(fake_fft, cmap="viridis")
            axes[1, 1].set_title("Deepfake 2D FFT Spectrum (Grid Artifacts)")
            axes[1, 1].axis("off")
            fig.colorbar(im1, ax=axes[1, 1])

            plt.tight_layout()
            plt.show()
except Exception as e:
    print(f"FFT Visualizer Note: {e}")


## 6. Two-Phase Training & Evaluation

In [ ]:
# ── Training & Evaluation ───────────────────────────────────────────────────
from src.training.trainer import TwoPhaseTrainer, train_two_phase
from src.training.evaluator import evaluate_full_suite

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
model_dual = build_model(use_fft=True, device=device, pretrained=True, config=CFG)
model_dual, opt_thresh_dual = train_two_phase(model_dual, train_loader, val_loader)

# Execute Publication-Grade Evaluation Suite (Frame & Video AUC, EER, ECE Calibration)
eval_res = evaluate_full_suite(model_dual, test_loader, device=device, val_loader=val_loader)
print(eval_res.summary_markdown())

test_targets = eval_res.targets
test_probs = eval_res.probs
temperature = float(getattr(eval_res, 'temperature', 1.0))


### 6.1 Dual-Stream Grad-CAM Heatmap Grid Visualizer
Renders Spatial (`target_stream="spatial"`) vs Frequency (`target_stream="frequency"`) Grad-CAM heatmaps overlaying test face crops.

In [ ]:
# Dual-Stream Grad-CAM Visualizer Grid
try:
    from src.explainability.gradcam import PyTorchGradCAM
    sample_imgs, sample_labels = next(iter(test_loader))
    if sample_imgs.ndim == 5:
        sample_imgs = sample_imgs[:, 0]  # Take first frame if sequence
    
    sample_batch = sample_imgs[:4].to(device)
    with PyTorchGradCAM(model_dual, target_stream="spatial") as spatial_cam:
        spatial_heatmaps = spatial_cam.generate_heatmaps_batch(sample_batch)
    with PyTorchGradCAM(model_dual, target_stream="frequency") as freq_cam:
        freq_heatmaps = freq_cam.generate_heatmaps_batch(sample_batch)

    fig, axes = plt.subplots(len(sample_batch), 4, figsize=(14, 3 * len(sample_batch)))
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

    for idx in range(len(sample_batch)):
        img_np = (sample_batch[idx].cpu().numpy() * std + mean).clip(0, 1).transpose(1, 2, 0)
        img_uint8 = (img_np * 255).astype(np.uint8)
        lbl = "Fake" if sample_labels[idx].item() == 1 else "Real"

        sp_overlay = PyTorchGradCAM.overlay_heatmap(img_uint8, spatial_heatmaps[idx])
        fr_overlay = PyTorchGradCAM.overlay_heatmap(img_uint8, freq_heatmaps[idx])

        axes[idx, 0].imshow(img_uint8)
        axes[idx, 0].set_title(f"Target: {lbl}")
        axes[idx, 0].axis("off")

        axes[idx, 1].imshow(spatial_heatmaps[idx], cmap="jet")
        axes[idx, 1].set_title("Spatial Grad-CAM")
        axes[idx, 1].axis("off")

        axes[idx, 2].imshow(freq_heatmaps[idx], cmap="jet")
        axes[idx, 2].set_title("Frequency Grad-CAM")
        axes[idx, 2].axis("off")

        axes[idx, 3].imshow(sp_overlay)
        axes[idx, 3].set_title("Spatial Overlay")
        axes[idx, 3].axis("off")

    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Grad-CAM Visualizer Note: {e}")


### 6.2 Confusion Matrix & ROC Curve Plots
Renders 2D Confusion Matrix heatmap and publication-grade ROC Curve with Equal Error Rate (EER) point.

In [ ]:
# Render Publication-Quality Confusion Matrix & ROC Curve
try:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # 1. Confusion Matrix
    cm = confusion_matrix(test_targets, (np.array(test_probs) >= opt_thresh_dual).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax1, xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])
    ax1.set_title("Confusion Matrix (Test Set)")
    ax1.set_xlabel("Predicted Label")
    ax1.set_ylabel("True Label")

    # 2. ROC Curve
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(test_targets, test_probs)
    ax2.plot(fpr, tpr, color="#2563eb", lw=2, label=f"Dual-Stream (AUC = {eval_res.auc:.4f})")
    ax2.plot([0, 1], [0, 1], color="#94a3b8", lw=1, linestyle="--")
    ax2.scatter([eval_res.eer], [1.0 - eval_res.eer], color="#dc2626", zorder=5, label=f"EER = {eval_res.eer*100:.2f}%")
    ax2.set_title("ROC Curve (Test Set)")
    ax2.set_xlabel("False Positive Rate (FPR)")
    ax2.set_ylabel("True Positive Rate (TPR)")
    ax2.legend(loc="lower right")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Evaluation Plotter Note: {e}")


## 7. Frequency Stream Ablation Study

In [ ]:
# Ablation study: Spatial-Only vs Dual-Stream
# Both models seeded identically for reproducibility.
# Note: This is a single-seed run. For publication, repeat ≥3 seeds and
# report mean ± std — a single delta can be within noise.
print("Ablation Study: Spatial-Only vs Dual-Stream")

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
model_spatial = build_model(use_fft=False)
model_spatial, opt_thresh_spatial = train_two_phase(model_spatial, train_loader, val_loader)

# Param count comparison
n_spatial = sum(p.numel() for p in model_spatial.parameters())
n_dual    = sum(p.numel() for p in model_dual.parameters())
print(f"Params: Spatial={n_spatial/1e6:.1f}M | Dual={n_dual/1e6:.1f}M "
      f"| FFT branch adds {(n_dual - n_spatial)/1e6:.2f}M params")
print("AUC delta includes extra capacity from FFT branch — not purely information gain.")

model_spatial.eval()
spatial_probs = []
with torch.no_grad():
    for imgs, _ in test_loader:
        imgs = imgs.to(device)
        device_type = "cuda" if device.type == "cuda" else "cpu"
        with torch.amp.autocast(device_type):
            p1 = torch.sigmoid(model_spatial(imgs))
            p2 = torch.sigmoid(model_spatial(torch.flip(imgs, dims=[-1])))
            probs = (p1 + p2) / 2.0
        spatial_probs.extend(probs.cpu().numpy())
        
spatial_auc = roc_auc_score(test_targets, spatial_probs)
spatial_acc = np.mean((np.array(spatial_probs) >= 0.5) == np.array(test_targets))
dual_auc = roc_auc_score(test_targets, test_probs)
dual_acc = np.mean((np.array(test_probs) >= 0.5) == np.array(test_targets))

print("\nAblation Results:")
print(f"Spatial-Only (ConvNeXt):      Acc = {spatial_acc*100:.2f}% | AUC = {spatial_auc:.4f}")
print(f"Dual-Stream (ConvNeXt + FFT): Acc = {dual_acc*100:.2f}% | AUC = {dual_auc:.4f}")


## 8. Leave-One-Type-Out (LOTO) Benchmark

In [ ]:
# ── Leave-One-Type-Out (LOTO) Generalization Benchmark ──────────────────────
# Correct methodology: FaceShifter is completely excluded from the split graph.
# A fresh identity-safe split is built on the remaining manipulation types.
# Test set = held-out test-split real faces + ALL FaceShifter samples (never seen).
# This guarantees zero FaceShifter identity appears in any training step.
print(f"LOTO Benchmark: Holding out '{HELD_OUT_TYPE}'")

# Step 1: Partition all_samples by manipulation type BEFORE any split
held_out_samples = [s for s in all_samples if HELD_OUT_TYPE.lower() in s[0].lower()]
non_held_samples  = [s for s in all_samples if HELD_OUT_TYPE.lower() not in s[0].lower()]
print(f"Held-out '{HELD_OUT_TYPE}': {len(held_out_samples)} samples (never in training)")
print(f"Non-held samples available for LOTO train/val/test: {len(non_held_samples)}")

# Step 2: Fresh identity-safe split on non-held-out data only
loto_train_s, loto_val_s, loto_test_s = perform_graph_split(non_held_samples, seed=SEED)

# Step 3: LOTO test = real faces from the held-out test split + ALL FaceShifter
loto_test_real = [s for s in loto_test_s if s[1] == 0]
loto_test_samples = loto_test_real + held_out_samples
print(f"LOTO | Train: {len(loto_train_s)} | Val: {len(loto_val_s)} "
      f"| Test: {len(loto_test_samples)} (Real={len(loto_test_real)}, "
      f"FaceShifter={len(held_out_samples)})")

loto_train_labels = [s[1] for s in loto_train_s]
loto_counts = np.maximum(np.bincount(loto_train_labels), 1)
loto_weights = 1. / loto_counts
loto_sample_weights = [loto_weights[l] for l in loto_train_labels]
loto_generator = torch.Generator().manual_seed(SEED)
loto_sampler = WeightedRandomSampler(
    weights=loto_sample_weights, num_samples=len(loto_sample_weights), replacement=True, generator=loto_generator)

loto_train_loader = DataLoader(DeepfakeDataset(loto_train_s, train_transform),
                               batch_size=batch_sz, sampler=loto_sampler,
                               num_workers=4, pin_memory=True, drop_last=True)
loto_val_loader   = DataLoader(DeepfakeDataset(loto_val_s, eval_transform),
                               batch_size=batch_sz, shuffle=False, num_workers=4, pin_memory=True)
loto_test_loader  = DataLoader(DeepfakeDataset(loto_test_samples, eval_transform),
                               batch_size=batch_sz, shuffle=False, num_workers=4, pin_memory=True)

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
model_loto = build_model(use_fft=True)
model_loto, opt_thresh_loto = train_two_phase(model_loto, loto_train_loader, loto_val_loader)

device_type = "cuda" if device.type == "cuda" else "cpu"
model_loto.eval()
loto_probs, loto_targets = [], []
with torch.no_grad():
    for imgs, labels in loto_test_loader:
        imgs = imgs.to(device)
        with torch.amp.autocast(device_type):
            p1 = torch.sigmoid(model_loto(imgs))
            p2 = torch.sigmoid(model_loto(torch.flip(imgs, dims=[-1])))
            probs = (p1 + p2) / 2.0
        loto_probs.extend(probs.cpu().numpy())
        loto_targets.extend(labels.cpu().numpy())

loto_probs_arr  = np.array(loto_probs)
loto_targets_arr = np.array(loto_targets)
loto_auc = roc_auc_score(loto_targets_arr, loto_probs_arr) if len(np.unique(loto_targets_arr)) > 1 else 0.5

# Primary metric at T=0.5 (honest — opt_thresh was calibrated on non-FaceShifter val)
loto_acc = np.mean((loto_probs_arr >= 0.5).astype(int) == loto_targets_arr)
print(f"Unseen Manipulation '{HELD_OUT_TYPE}' | AUC: {loto_auc:.4f} | Acc (T=0.5): {loto_acc*100:.2f}%")
try:
    eer_loto = calculate_crash_proof_eer(loto_targets_arr, loto_probs_arr)
    print(f"LOTO Equal Error Rate (EER): {eer_loto * 100:.2f}%")
except Exception as e:
    print(f"EER Note: {e}")


## 9. Celeb-DF v2 Cross-Dataset Benchmark Evaluation

In [ ]:
# Dynamic wildcard discovery for Reuben Suju's Celeb-DF v2 MP4 videos
print("--- Celeb-DF v2 Cross-Dataset Benchmark Evaluation ---")
real_dirs = glob.glob("/kaggle/input/**/Celeb-real", recursive=True)
fake_dirs = glob.glob("/kaggle/input/**/Celeb-synthesis", recursive=True)

if real_dirs and fake_dirs:
    celeb_real_mp4s = glob.glob(f"{real_dirs[0]}/*.mp4")
    celeb_fake_mp4s = glob.glob(f"{fake_dirs[0]}/*.mp4")
    print(f"Celeb-DF v2 (Reuben Suju) Detected | Real MP4s: {len(celeb_real_mp4s)} | Fake MP4s: {len(celeb_fake_mp4s)}")

    celeb_crop_dir = "/tmp/celeb_df_ram"
    cropper = DynamicFaceCropper(target_size=256, scale_factor=1.30, device=device)
    if not os.path.exists(celeb_crop_dir):
        os.makedirs(f"{celeb_crop_dir}/real", exist_ok=True)
        os.makedirs(f"{celeb_crop_dir}/fake", exist_ok=True)
        for vid in tqdm(celeb_real_mp4s, desc="Cropping Celeb-Real MP4s"):
            cropper.extract_faces_from_video(vid, f"{celeb_crop_dir}/real", max_frames=10)
        for vid in tqdm(celeb_fake_mp4s, desc="Cropping Celeb-Synthesis MP4s"):
            cropper.extract_faces_from_video(vid, f"{celeb_crop_dir}/fake", max_frames=10)

    celeb_real_crops = glob.glob(f"{celeb_crop_dir}/real/*.jpg")
    celeb_fake_crops = glob.glob(f"{celeb_crop_dir}/fake/*.jpg")
    celeb_samples = [(f, 0) for f in celeb_real_crops] + [(f, 1) for f in celeb_fake_crops]
    celeb_loader = DataLoader(DeepfakeDataset(celeb_samples, eval_transform), batch_size=32, shuffle=False, num_workers=4)

    model_dual.eval()
    celeb_probs, celeb_targets = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(celeb_loader, desc="Evaluating Celeb-DF v2 Transfer"):
            imgs = imgs.to(device)
            device_type = "cuda" if device.type == "cuda" else "cpu"
            with torch.amp.autocast(device_type):
                p1 = torch.sigmoid(model_dual(imgs))
                p2 = torch.sigmoid(model_dual(torch.flip(imgs, dims=[-1])))
                probs = (p1 + p2) / 2.0
            celeb_probs.extend(probs.cpu().numpy())
            celeb_targets.extend(labels.cpu().numpy())

    celeb_probs_arr = np.array(celeb_probs)
    celeb_targets_arr = np.array(celeb_targets)
    celeb_auc = roc_auc_score(celeb_targets_arr, celeb_probs_arr)
    celeb_acc = np.mean((celeb_probs_arr >= 0.5).astype(int) == celeb_targets_arr)
    print(f"\n--- Celeb-DF v2 Cross-Dataset Benchmark Results ---")
    print(f"Cross-Dataset Transfer AUC: {celeb_auc:.4f} | Accuracy (T=0.5): {celeb_acc*100:.2f}%")
    try:
        from src.training.evaluator import calculate_crash_proof_eer
        eer_c = calculate_crash_proof_eer(celeb_targets_arr, celeb_probs_arr)
        print(f"Celeb-DF v2 Equal Error Rate (EER): {eer_c * 100:.2f}%")
    except Exception as e:
        print(f"EER Note: {e}")
else:
    print("Celeb-DF v2 dataset not mounted. To run cross-dataset transfer, attach 'celeb-df-v2' in Kaggle input.")


## 10. Save Checkpoint & ONNX Export

In [ ]:
# Save model checkpoint and export to ONNX format
export_pth_path = "/kaggle/working/deepfake_convnext_v2.pth"
unwrapped_model = model_dual.module if hasattr(model_dual, 'module') else model_dual
checkpoint = {
    'state_dict': unwrapped_model.state_dict(),
    'optimal_threshold': float(opt_thresh_dual),
    'temperature': float(temperature if 'temperature' in locals() else 1.0),
    'val_auc': float(test_auc),
    'config': CFG
}
torch.save(checkpoint, export_pth_path)
print(f"Saved PyTorch checkpoint with optimal_threshold ({opt_thresh_dual:.4f}): {export_pth_path}")

try:
    from src.models.onnx_exporter import export_to_onnx
    export_onnx_path = "/kaggle/working/deepfake_convnext_v2.onnx"
    export_to_onnx(unwrapped_model, save_path=export_onnx_path, img_size=IMG_SIZE)
    print(f"Exported ONNX model: {export_onnx_path}")
except Exception as e:
    print(f"ONNX export failed: {e}")


### 10.1 PyTorch CUDA vs ONNX Runtime Latency & FPS Benchmark
Measures PyTorch FP32, PyTorch AMP FP16, and ONNX Runtime execution speeds in ms/frame and FPS.

In [ ]:
# Inference Latency & FPS Benchmark: PyTorch vs ONNX Runtime
try:
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    dummy_numpy = dummy_input.cpu().numpy()
    warmup = 10
    iters = 50

    # PyTorch FP32 Timing
    model_dual.eval()
    with torch.no_grad():
        for _ in range(warmup):
            _ = model_dual(dummy_input)
        
        t0 = time.perf_counter()
        for _ in range(iters):
            _ = model_dual(dummy_input)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()
    
    pt_latency = ((t1 - t0) / iters) * 1000.0
    pt_fps = 1000.0 / pt_latency
    print(f"PyTorch FP32 Latency: {pt_latency:.2f} ms/frame | Throughput: {pt_fps:.1f} FPS")

    # PyTorch AMP FP16 Timing
    with torch.no_grad():
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            for _ in range(warmup):
                _ = model_dual(dummy_input)
            
            t0 = time.perf_counter()
            for _ in range(iters):
                _ = model_dual(dummy_input)
            if device.type == "cuda":
                torch.cuda.synchronize()
            t1 = time.perf_counter()
    
    amp_latency = ((t1 - t0) / iters) * 1000.0
    amp_fps = 1000.0 / amp_latency
    print(f"PyTorch AMP FP16 Latency: {amp_latency:.2f} ms/frame | Throughput: {amp_fps:.1f} FPS")

    # ONNX Runtime Timing
    if os.path.exists(export_onnx_path):
        import onnxruntime as ort
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if torch.cuda.is_available() else ["CPUExecutionProvider"]
        session = ort.InferenceSession(export_onnx_path, providers=providers)
        input_name = session.get_inputs()[0].name
        
        for _ in range(warmup):
            _ = session.run(None, {input_name: dummy_numpy})
        
        t0 = time.perf_counter()
        for _ in range(iters):
            _ = session.run(None, {input_name: dummy_numpy})
        t1 = time.perf_counter()
        
        onnx_latency = ((t1 - t0) / iters) * 1000.0
        onnx_fps = 1000.0 / onnx_latency
        print(f"ONNX Runtime Latency:  {onnx_latency:.2f} ms/frame | Throughput: {onnx_fps:.1f} FPS")
except Exception as e:
    print(f"Latency Benchmark Note: {e}")
